In [1]:
!pip install -q sdv optuna pandas wandb optuna-integration

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.9/157.9 kB 3.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.5/98.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.3/69.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.4/193.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 34.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 M

In [2]:
!pip install optbinning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.9/213.9 kB 4.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 60.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 16.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.26.

In [3]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from optbinning import OptimalBinning

# ✅ Step 1: Load your dataset
print("📂 Loading dataset...")
df = pd.read_csv('/kaggle/input/customer-churn-openml-2-7043/dataset1_openml_7043x21.csv')
print(f"✅ Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.\n")

# ✅ Step 2: Only 'tenure' will be binned
numerical_feature_to_bin = ['tenure']
print(f"🎯 Numerical feature to bin: {numerical_feature_to_bin}\n")

# ✅ Step 3: Drop 'customerID' column if it exists
if 'customerID' in df.columns:
    print("🗑️ Dropping 'customerID' column...")
    df = df.drop(columns=['customerID'])
    print("✅ 'customerID' column dropped.\n")
else:
    print("ℹ️ 'customerID' column not found, skipping drop.\n")

# ✅ Step 4: Convert 'Churn' column to binary
if df['Churn'].dtype == object:
    print("🔄 Converting 'Churn' values to 1/0...")
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
    print("✅ 'Churn' conversion complete.\n")
else:
    print("ℹ️ 'Churn' column already numeric, no conversion needed.\n")

# ✅ Step 5: Handle bad/missing values in 'TotalCharges'
print("🔍 Checking and cleaning 'TotalCharges' column...")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing_count = df['TotalCharges'].isna().sum()
print(f"⚠️ Found {missing_count} missing/invalid values in 'TotalCharges'. Dropping these rows...")

df = df.dropna(subset=['TotalCharges']).reset_index(drop=True)
print(f"✅ Dropped rows with missing TotalCharges. New dataset shape: {df.shape}\n")

# ✅ Step 6: Create a copy to store binned versions
df_binned = df.copy()
print("📝 Created a copy of the dataset for binning.\n")

# ✅ Step 7: Perform Optimal Binning only for 'tenure'
for idx, feature in enumerate(numerical_feature_to_bin, start=1):
    print(f"🚀 [{idx}/{len(numerical_feature_to_bin)}] Binning feature: '{feature}'...")

    optb = OptimalBinning(
        name=feature,
        dtype="numerical",
        solver="cp",
        max_n_bins=6,
        min_bin_size=0.05
    )

    print(f"🔧 Fitting OptimalBinning model on '{feature}'...")
    optb.fit(df[feature], df['Churn'])
    print(f"✅ Fitting complete for '{feature}'.")

    binning_table = optb.binning_table.build()
    print(f"📊 Binning Table for '{feature}':")
    print(binning_table, "\n")

    print(f"🎨 Transforming '{feature}' into categorical bins...")
    df_binned[feature + "_binned"] = optb.transform(df[feature], metric="bins")
    print(f"✅ Transformation complete for '{feature}'.\n")

# ✅ Step 8: Prepare datasets
binned_features = [col + "_binned" for col in numerical_feature_to_bin]

# In the binned dataset:
# - Use 'tenure_binned' instead of 'tenure'
# - Keep 'MonthlyCharges' and 'TotalCharges' as they are
X_original = df.drop(columns=['Churn'])
X_binned = df_binned.drop(columns=['tenure', 'Churn'])
y = df['Churn']

print("✅ Prepared both original and partially binned datasets.\n")

# ✅ Step 9: Label Encode categorical features
print("🔠 Label encoding categorical features in original dataset...")
categorical_cols_orig = X_original.select_dtypes(include=['object']).columns.tolist()
print(f"ℹ️ Categorical columns found in original data: {categorical_cols_orig}")

le = LabelEncoder()
for col in categorical_cols_orig:
    X_original[col] = le.fit_transform(X_original[col])

print("✅ Label encoding completed for original dataset.\n")

print("🔠 Label encoding categorical features in binned dataset...")
categorical_cols_binned = X_binned.select_dtypes(include=['object']).columns.tolist()
print(f"ℹ️ Categorical columns found in binned data: {categorical_cols_binned}")

for col in categorical_cols_binned:
    X_binned[col] = le.fit_transform(X_binned[col])

print("✅ Label encoding completed for binned dataset.\n")

# ✅ Step 10: Train-test split
print("✂️ Splitting data into train and test sets...\n")
X_train_orig, X_test_orig, y_train, y_test = train_test_split(X_original, y, test_size=0.2, random_state=42, stratify=y)
X_train_binned, X_test_binned, _, _ = train_test_split(X_binned, y, test_size=0.2, random_state=42, stratify=y)

# ✅ Step 11: Handle class imbalance
print("⚖️ Handling class imbalance...")
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f"✅ Computed scale_pos_weight: {scale_pos_weight:.4f}\n")

# ✅ Step 12: Train LightGBM models
print("🚀 Training LightGBM model on original features...")
model_orig = lgb.LGBMClassifier(random_state=42, scale_pos_weight=scale_pos_weight)
model_orig.fit(X_train_orig, y_train)

print("🚀 Training LightGBM model on partially binned features...")
model_binned = lgb.LGBMClassifier(random_state=42, scale_pos_weight=scale_pos_weight)
model_binned.fit(X_train_binned, y_train)

# ✅ Step 13: Evaluate models
print("\n📊 Evaluating models...\n")

def evaluate_model(model, X_test, y_test, title="Model"):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:,1]
    print(f"📋 Evaluation Results for {title}:")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("Precision:", precision_score(y_test, preds))
    print("Recall:", recall_score(y_test, preds))
    print("F1 Score:", f1_score(y_test, preds))
    print("ROC-AUC:", roc_auc_score(y_test, probs))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))
    print("\nDetailed Classification Report:\n", classification_report(y_test, preds))
    print("-" * 60)

# ✅ Evaluate both models
evaluate_model(model_orig, X_test_orig, y_test, title="Original Features")
evaluate_model(model_binned, X_test_binned, y_test, title="Partially Binned Features (only 'tenure')")


(CVXPY) Apr 28 02:52:54 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Apr 28 02:52:54 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')
📂 Loading dataset...
✅ Dataset loaded with 7043 rows and 21 columns.

🎯 Numerical feature to bin: ['tenure']

🗑️ Dropping 'customerID' column...
✅ 'customerID' column dropped.

🔄 Converting 'Churn' values to 1/0...
✅ 'Churn' conversion complete.

🔍 Checking and cleaning 'TotalCharges' column...
⚠️ Found 11 missing/invalid values in 'TotalCharges'. Dropping these rows...
✅ Dropped rows with missing TotalCharges. New dataset shape: (7032, 20)

📝 Created a copy of the dataset for binning.

🚀 [1/1] Binning featur

In [4]:
# ✅ Step 14: Feature Importance for 'model_binned'

print("\n📈 Calculating feature importances for partially binned model...")

# Get feature importances from the LightGBM model
importances = model_binned.feature_importances_

# Normalize to percentages
importances_percent = 100.0 * (importances / importances.sum())

# Get feature names
feature_names = X_train_binned.columns

# Create and print feature importance percentages
print("\n🔍 Feature Importance Percentages:")
for feature, importance in zip(feature_names, importances_percent):
    print(f"- {feature}: {importance:.2f}%")

# ✅ (Optional) If you want them sorted from highest to lowest:
sorted_features = sorted(zip(feature_names, importances_percent), key=lambda x: x[1], reverse=True)

print("\n🏆 Features sorted by importance (highest to lowest):")
for feature, importance in sorted_features:
    print(f"- {feature}: {importance:.2f}%")



📈 Calculating feature importances for partially binned model...

🔍 Feature Importance Percentages:
- gender: 3.57%
- SeniorCitizen: 1.70%
- Partner: 1.77%
- Dependents: 1.97%
- PhoneService: 0.40%
- MultipleLines: 2.17%
- InternetService: 1.03%
- OnlineSecurity: 2.40%
- OnlineBackup: 2.50%
- DeviceProtection: 1.27%
- TechSupport: 2.43%
- StreamingTV: 1.30%
- StreamingMovies: 1.97%
- Contract: 3.53%
- PaperlessBilling: 2.67%
- PaymentMethod: 6.23%
- MonthlyCharges: 28.73%
- TotalCharges: 29.83%
- tenure_binned: 4.53%

🏆 Features sorted by importance (highest to lowest):
- TotalCharges: 29.83%
- MonthlyCharges: 28.73%
- PaymentMethod: 6.23%
- tenure_binned: 4.53%
- gender: 3.57%
- Contract: 3.53%
- PaperlessBilling: 2.67%
- OnlineBackup: 2.50%
- TechSupport: 2.43%
- OnlineSecurity: 2.40%
- MultipleLines: 2.17%
- Dependents: 1.97%
- StreamingMovies: 1.97%
- Partner: 1.77%
- SeniorCitizen: 1.70%
- StreamingTV: 1.30%
- DeviceProtection: 1.27%
- InternetService: 1.03%
- PhoneService: 0.40%


In [5]:
# 📦 Import libraries
import pandas as pd
import numpy as np
import torch
import random
import time
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality
import warnings
warnings.filterwarnings("ignore")

# ✅ Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ✅ Start timing
start_time = time.time()
print("🚀 Starting reproducible CTGAN generation on GPU...\n")

# ✅ Load the processed dataset (binned 'tenure', others untouched)
print("📂 Loading processed dataset (binned 'tenure' only)...")
# You can replace the below path if needed, or directly use df_binned created earlier if running in one script.
# If you saved it somewhere, you would load that CSV; else, use df_binned.
# For now, assuming we have df_binned from your previous code
df_binned_with_churn = X_binned.copy()
df_binned_with_churn['Churn'] = y.values
print("✅ Dataset loaded. Shape:", df_binned_with_churn.shape, "\n")

# ✅ Print first two rows
print("🧾 First two rows of dataset to be used for CTGAN training:")
print(df_binned_with_churn.head(2))
print("\n✅ Dataset ready for synthetic data generation.\n")

# ✅ Define metadata
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_binned_with_churn)
print(f"✅ Metadata detection complete. Dataset shape: {df_binned_with_churn.shape}\n")

# ✅ Best CTGAN Hyperparameters (as per your earlier setup)
best_params = {
    'batch_size': 360,
    'epochs': 1552,
    'embedding_dim': 110,
    'generator_lr': 0.004713204343061388,
    'discriminator_lr': 0.0017260261506771618,
    'generator_dim': (299, 216),
    'discriminator_dim': (230, 207),
    'verbose': True,
    'cuda': torch.cuda.is_available()
}

# ✅ Train CTGAN
print("🧠 Training CTGAN with best parameters on CUDA..." if best_params['cuda'] else "🧠 Training CTGAN with best parameters on CPU...")
train_start = time.time()
model = CTGANSynthesizer(metadata, **best_params)
model.fit(df_binned_with_churn)
train_end = time.time()
print(f"⏱️ Training time: {train_end - train_start:.2f} seconds\n")

# ✅ Generate synthetic data
print("📈 Generating synthetic data (70,000 rows)...")
generate_start = time.time()
synthetic_df = model.sample(70000)
generate_end = time.time()
print(f"⏱️ Generation time: {generate_end - generate_start:.2f} seconds\n")

# ✅ Save synthetic data
output_path = "/kaggle/working/augmented_CTGAN_Binned_Tenure_Only_Churn_dataset.csv"
synthetic_df.to_csv(output_path, index=False)
print(f"📁 Synthetic data saved to: {output_path}\n")

# ✅ Completion
end_time = time.time()
print(f"🎉 Completed in {end_time - start_time:.2f} seconds")


🚀 Starting reproducible CTGAN generation on GPU...

📂 Loading processed dataset (binned 'tenure' only)...
✅ Dataset loaded. Shape: (7032, 20) 

🧾 First two rows of dataset to be used for CTGAN training:
   gender  SeniorCitizen  Partner  Dependents  PhoneService  MultipleLines  \
0       0              0        1           0             0              1   
1       1              0        0           0             1              0   

   InternetService  OnlineSecurity  OnlineBackup  DeviceProtection  \
0                0               0             2                 0   
1                0               2             0                 2   

   TechSupport  StreamingTV  StreamingMovies  Contract  PaperlessBilling  \
0            0            0                0         0                 1   
1            0            0                0         1                 0   

   PaymentMethod  MonthlyCharges  TotalCharges  tenure_binned  Churn  
0              2           29.85         29.85     

Gen. (-0.39) | Discrim. (-0.19): 100%|██████████| 1552/1552 [50:10<00:00,  1.94s/it]


⏱️ Training time: 3022.99 seconds

📈 Generating synthetic data (70,000 rows)...
⏱️ Generation time: 6.20 seconds

📁 Synthetic data saved to: /kaggle/working/augmented_CTGAN_Binned_Tenure_Only_Churn_dataset.csv

🎉 Completed in 3029.66 seconds


In [6]:
# 📦 Save the trained CTGAN model
model_save_path = "/kaggle/working/ctgan_model_2_only_tenure_categorical.pkl"

# ✅ Save the model using SDV's save method
model.save(model_save_path)
print(f"✅ CTGAN model saved to: {model_save_path}")

# 📥 To download the model in Kaggle Notebooks, create a download link:
from IPython.display import FileLink

print("\n📁 Click below link to download the saved CTGAN model:")
display(FileLink(model_save_path))


✅ CTGAN model saved to: /kaggle/working/ctgan_model_2_only_tenure_categorical.pkl

📁 Click below link to download the saved CTGAN model:


/kaggle/working/ctgan_model_2_only_tenure_categorical.pkl